In [0]:
catalog = 'kfcus_digital_model'
schema = 'recommender_test'

In [0]:
sdf_cleaned_mapped = spark.read.table(f'{catalog}.{schema}.cleaned_mapped_dataset')

## Evaluate support of products. 
Support is the proportion of transactions an item shows up in. The values range from 0 to 1; a greater support means a larger number of transactions includes that particular item (e.g. 0.8 support for Biscuits means that Biscuits shows up in 80% of the transactions).

In [0]:
from pyspark.sql.functions import explode, col, count

# Explode the order_product_list and create columns for count and proportion of each product in all orders
sdf_unique_count = sdf_cleaned_mapped.withColumn('order_product', explode(col('order_product_list'))) \
                              .groupBy('order_product') \
                              .agg(count('*').alias('count'), (count('*') / sdf_cleaned_mapped.count()).alias('support')) \
                              .sort(col('count').desc())

df_eda = sdf_unique_count.toPandas()
sdf_unique_count.display()

In [0]:
import matplotlib.pyplot as plt

top_items = df_eda.head(25)
plt.figure(figsize=(8, 4))
plt.bar(top_items['order_product'], top_items['support']*100)
plt.xlabel('Product Slug')
plt.ylabel('Support (i.e. percentage of orders (%))')
plt.title('Top 25 Ordered Products')
plt.xticks(rotation=90)
plt.show()

# Perform our Market Basket Analysis

A market basket analysis requires creating a one-hot encoding for our item set x transaction ID before leveraging the apriori algorithm to generate association rules. To improve computational efficiency, we must set a threshold for `support` and `confidence`. 
* `Support` is the proportion of transactions that an item shows up in. The values range from 0 to 1, where 0 means an item never shows up in any transaction and 1 meaning it shows up in every transaction.
* `Confidence` is the frequency with which the consequent (another item) occurs whenever the antecedent (the items in the cart) is present. The values range from 0 to 1; 0 means the items are never seen together and 1 means they are always seeen together. 



## MBA using PySpark's FPGrowth

In [0]:
from pyspark.ml.fpm import FPGrowth

# We can define a support threshold via the total number of transactions that an item or item combination has to show up in the dataset to be considered frequent for rule-mining.
min_transactions = 1000
num_of_transactions = sdf_cleaned_mapped.count() 
# This will give us MANY recommendations for some but will ensure we have at least something to recommend. Having too many recommendations can be problematic since the function itself does not return a confidence for each recommendation.
min_confidence = 0.00

# configure model
fpGrowth = FPGrowth(
              itemsCol='order_product_list', 
              minSupport=min_transactions/num_of_transactions, 
              minConfidence=min_confidence,
              numPartitions=sc.defaultParallelism * 100
              )
 
# train_test split for dataset
train, test = sdf_cleaned_mapped.randomSplit(weights=[0.8, 0.2], seed=42)

# fit model to item set data
model = fpGrowth.fit(train)
 
# count number of rules generated
model.associationRules.count()
model.associationRules.sort("antecedent", "consequent").display()

In [0]:
# Save model rules, train, and test dataset to UC
model.associationRules.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{catalog}.{schema}.assoc_rules')

## Test + Evaluation
Switching from personal compute with single-node to shared compute with multi-node since we won't have to use Spark ML here. Purely doing inferencing by joining and manipulating tables of rules + carts.

In [0]:
from pyspark.sql.functions import broadcast, array_intersect, array, expr, size, array_intersect, power, row_number, collect_list, struct
from pyspark.sql.window import Window

def generate_recommendations(rules_sdf, cart_sdf, k=5, metric='confidence', testing=False):
    """
    Generate top-k product recommendations based on association rules.

    Parameters:
    rules_sdf (DataFrame): Spark DataFrame containing association rules generated from FPGrowth with columns 'antecedent', 'consequent', 'lift', and 'confidence'.
    cart_sdf (DataFrame): Spark DataFrame containing cart data with columns 'order_id', 'cart', and optionally 'added' if testing.
    k (int): Number of top recommendations to return. Default is 5.
    metric (str): Metric to use for scoring rules. Default is 'confidence'.
    testing (bool): If True, include 'added' column in cart_sdf. Output dataframe will pass this column on as well. Default is faulse

    Returns:
    DataFrame: Spark DataFrame with columns 'order_id', 'cart', and 'recommendations_with_scores' containing top-k recommendations. Additionally will have 'added' column if testing is True.
    """
    # Match all cart items with rules. In short, this looks at the cart and finds any rules with associations. 
    baskets_and_rules = (
      cart_sdf
        .join(
          broadcast(
            rules_sdf.selectExpr('antecedent', 'consequent', 'lift', 'confidence')
          ),
          on=array_intersect(col('cart'), col('antecedent')) != array()
        )
    )

    # Create a rule score using confidence that places more weight on rules with higher matches of rules antecedent with cart. 
    score = (
      baskets_and_rules
        .filter(expr('not array_contains(cart, consequent)'))  # consequent not already in cart
        .withColumn('intr', expr('size(array_intersect(cart, antecedent))'))
        .withColumn('match_score', expr('power(intr, 2) / (size(antecedent) * size(cart))'))
        .withColumn('rule_score', expr(f'{metric} * match_score'))
    )

    # If testing, there should be an 'added' column to the cart_sdf that will have to be included. 
    return_columns = ["order_id", "cart"]
    if testing:
      return_columns += ['added']
      
    # Get top consequent score and grab the recommend the top k items.
    top_k_recommendations = (
        score.withColumn("rank", row_number().over(Window.partitionBy("order_id", "consequent").orderBy(col("rule_score").desc())))
                .filter(col("rank") == 1)
                .withColumn("rank", row_number().over(Window.partitionBy("order_id").orderBy(col("rule_score").desc())))
                .filter(col("rank") <= k)
                .groupBy(return_columns)
                .agg(
                    collect_list(struct("consequent", "rule_score")).alias("recommendations_with_scores")
                )
    )
    
    return top_k_recommendations
  


In [0]:
from pyspark.sql.functions import expr, size, col

# Load rules
catalog = 'kfcus_digital_model'
schema = 'recommender_test'

rules = (
  spark.read.table(f'{catalog}.{schema}.assoc_rules')
    .selectExpr('antecedent', 'consequent[0] as consequent', 'lift', 'confidence')
  )

# Load test dataset
test  = (
    spark.read.table(f'{catalog}.{schema}.test_dataset')
    .selectExpr('mpid', 'order_id', 'order_product_list')
).sample(withReplacement=False, fraction=0.50).repartition(sc.defaultParallelism*10)

# Remove rows with only one item in order_product_list
test_filtered = test.filter(size(col("order_product_list")) > 1)
# Create "cart" and "added" columns. Cart is all items with the last item in the list removed. Added is the last item in the list.
test_transformed = test_filtered.withColumn("cart", expr("slice(order_product_list, 1, size(order_product_list) - 1)")) \
                                .withColumn("added", expr("order_product_list[size(order_product_list) - 1]"))
test_set_count = test.count()
print(f'{test_set_count - test_filtered.count()} rows out of {test_set_count} remain in test after filtering out rows with only one item')

In [0]:
from pyspark.sql.functions import expr, array_contains, slice

baseline_recommendations = ['5-piece-nuggets',
 'biscuit',
 'cherry-pie-poppers',
 'chocolate-chip-cake',
 'extra-crispy-chicken-little',
 'famous-bowl',
 'mashed-potatoes',
 'original-recipe-chicken-little',
 'secret-recipe-fries']

k=5
# Generate top k recommendations using our MBA rules
top_k_recommendations = generate_recommendations(rules, test_transformed, k=k, metric='confidence', testing=True)

# Evaluate performance of baseline recommendations
top_k_recommendations = top_k_recommendations.withColumn(
    "hit@k_mba", 
    expr("cast(array_contains(transform(recommendations_with_scores, x -> x.consequent), added) as int)")
).withColumn(
    "hit@k_baseline", 
    expr("cast(array_contains(slice(array(" + 
         ', '.join([f"'{item}'" for item in baseline_recommendations]) + 
         f"), 1, {k}), added) as int)")
)

# Display what proportion of time the MBA top k recommendations are added compared to the static recommendations
display(top_k_recommendations.agg({"hit@k_mba": "avg", "hit@k_baseline": "avg"}))

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import expr, size, col

# Load rules
catalog = 'kfcus_digital_model'
schema = 'recommender_test'

rules = (
  spark.read.table(f'{catalog}.{schema}.assoc_rules')
    .selectExpr('antecedent', 'consequent[0] as consequent', 'lift', 'confidence')
  )

cart = [Row(order_id='123456', cart=['biscuit', 'secret-recipe-fries'], added='')]
cart_df = spark.createDataFrame(cart)

top_k_recommendations = generate_recommendations(rules, cart_df, k=5, metric='confidence', testing=False)
top_k_recommendations.display()


## Deploy our model 
Deploy MBA via generating a pyfunc that reads in the association rules and generates recommendations based on that. 
1. Create a custom PyFunc for our MBA model
2. Log the model to Unity Catalog


In [0]:
import pandas as pd
import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.types import ColSpec, Schema, DataType
from mlflow.types.schema import Array

class RecommenderModel(mlflow.pyfunc.PythonModel):
    def __init__(self, k=5, metric='confidence'):
        self.k = k
        self.metric = metric
        
    def load_context(self, context):
        """Load rules from packaged artifact"""
        rules_path = context.artifacts["rules_file"]
        self.rules_df = pd.read_parquet(rules_path)

    def _calculate_match_score(self, row):
        intersection = set(row['cart']).intersection(row['antecedent'])
        match_score = (len(intersection)**2) / (len(row['antecedent']) * len(row['cart']))
        return match_score

    def predict(self, context, model_input):
        merged = model_input.merge(self.rules_df, how='cross')
        if merged.empty:
            return []
        merged = merged[merged.apply(lambda x: bool(set(x['cart']) & set(x['antecedent'])), axis=1)]

        merged['match_score'] = merged.apply(self._calculate_match_score, axis=1)
        merged['rule_score'] = merged[self.metric] * merged['match_score']
        merged = merged[~merged.apply(lambda x: x['consequent'] in x['cart'], axis=1)]

        # Group rules by order_id and consequent and select the rule_score that's highest for each consequent
        grouped = merged.loc[merged.groupby(['order_id', 'consequent'])['rule_score'].idxmax()]

        # Filter to top K rules
        top_k = grouped.groupby('order_id').apply(
            lambda x: x.nlargest(self.k, 'rule_score')
        ).reset_index(drop=True)

        # Create a final dataframe
        result = top_k.groupby('order_id').agg({
            'cart': 'first',
            'consequent': list,
            'rule_score': list
        }).reset_index()

        result.rename(columns={'consequent': 'recommendations'}, inplace=True)
        
        return result

In [0]:

model_name = 'mba_recommender_model'

# Save rules locally as parquet to package as artifact in logged model
rules = (
  spark.read.table(f'{catalog}.{schema}.assoc_rules')
    .selectExpr('antecedent', 'consequent[0] as consequent', 'lift', 'confidence')
  ).toPandas()
rules.to_parquet('rules_table.parquet')

# Set registry to UC so we can save model to UC
# mlflow.set_registry_uri('databricks-uc')

# Define input + output schema of model to create signature
input_schema = Schema([
    ColSpec(DataType.string, "order_id"),
    ColSpec(Array(DataType.string), "cart") 
])
output_schema = Schema([
    ColSpec(DataType.string, "order_id"),
    ColSpec(Array(DataType.string), "cart"),
    ColSpec(Array(DataType.string), "recommendations"),
    ColSpec(Array(DataType.string), "rule_score")
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

# Provide sample model input
model_input = pd.DataFrame({
"order_id": ["12345"],
"cart": [
    ['biscuit','secret-recipe-fries']
    ]}
)

# Create a tracking run in mlflow and log the model to UC with the packaged rules parquet
with mlflow.start_run(run_name='market_basket_analysis_250320'):
    mlflow.pyfunc.log_model(
        "model",
        python_model=RecommenderModel(),
        registered_model_name=f"{model_name}",
        artifacts={"rules_file": "rules_table.parquet"},
        input_example=model_input,
        signature=signature 
        )

In [0]:
# Use last active run to load in model URI
details = mlflow.last_active_run()
model_uri = f"runs:/{details.info.run_id}/model"

# Test inference by loading model from UC + generating recommendations
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Generate input dataframe
input_data = pd.DataFrame({
    "order_id": ["12345", '1234'],
    "cart": [
        ['biscuit','secret-recipe-fries'],
        ['biscuit']
    ]
})

# Make recommendations
recommendations = loaded_model.predict(input_data)

recommendations